In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
import pandas as pd
import torch
from transformers import AutoTokenizer,AutoModelForMultipleChoice,TrainingArguments,Trainer
from peft import LoraConfig,get_peft_model,TaskType
from datasets import Dataset
import torch.nn.functional as F

df=pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
label_map={"A":0,"B":1,"C":2,"D":3,"E":4}
df["label"]=df["answer"].map(label_map)
q1_answer=df.loc[150,"label"]
print("Q1 - label at index 150:",q1_answer)

Q1 - label at index 150: 2


In [3]:
row0=df.loc[0]
option_b_input=str(row0["prompt"])+" [SEP] "+str(row0["B"])
q2_answer=len(option_b_input)
print("Q2 - length of formatted option B input:",q2_answer)

Q2 - length of formatted option B input: 407


In [4]:
model_name="bert-base-uncased"
tokenizer=AutoTokenizer.from_pretrained(model_name)
 
def format_row(row):
    prompt=str(row["prompt"])
    options=[str(row["A"]),str(row["B"]),str(row["C"]),str(row["D"]),str(row["E"])]
    return [prompt+" [SEP] "+opt for opt in options]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [5]:
inputs_row0=format_row(row0)
enc=tokenizer(inputs_row0,padding="max_length",truncation=True,max_length=128,return_tensors="pt")
input_ids=enc["input_ids"].unsqueeze(0) # reshape to [1,5,128]
attention_mask=enc["attention_mask"].unsqueeze(0)
print("Q3 - input_ids shape:",input_ids.shape,"second dim:",input_ids.shape[1])

Q3 - input_ids shape: torch.Size([1, 5, 128]) second dim: 5


In [6]:
batch_df=df.iloc[:16]
all_inputs=[]
for _,r in batch_df.iterrows():
    all_inputs.extend(format_row(r))
 
batch_enc=tokenizer(all_inputs,padding="max_length",truncation=True,max_length=128,return_tensors="pt")
batch_input_ids=batch_enc["input_ids"].view(16,5,128)
total_positions=batch_input_ids.numel()
print("Q4 - batch input_ids shape:",batch_input_ids.shape,"total token positions:",total_positions)

Q4 - batch input_ids shape: torch.Size([16, 5, 128]) total token positions: 10240


In [7]:
mc_model=AutoModelForMultipleChoice.from_pretrained(model_name)
mc_model.eval()
with torch.no_grad():
    outputs=mc_model(input_ids=input_ids,attention_mask=attention_mask)
logits=outputs.logits
print("Q5 - logits shape:",logits.shape,"num logits per question:",logits.shape[1])

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Q5 - logits shape: torch.Size([1, 5]) num logits per question: 5


In [8]:
label_row0=torch.tensor([row0["label"]])
with torch.no_grad():
    outputs_loss=mc_model(input_ids=input_ids,attention_mask=attention_mask,labels=label_row0)
loss=outputs_loss.loss
print("Q6 - loss tensor shape:",loss.shape,"num dims:",loss.dim())

Q6 - loss tensor shape: torch.Size([]) num dims: 0


In [10]:
!pip uninstall -y torchao -q

In [11]:
lora_config=LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query","value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)
lora_model=get_peft_model(AutoModelForMultipleChoice.from_pretrained(model_name),lora_config)
trainable_params=sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
print("Q7 - trainable params:",trainable_params)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Q7 - trainable params: 295681


In [12]:
subset_df=df.iloc[:100].reset_index(drop=True)
 
def build_example(row):
    texts=format_row(row)
    enc=tokenizer(texts,padding="max_length",truncation=True,max_length=128)
    return {
        "input_ids":enc["input_ids"],          # shape [5,128]
        "attention_mask":enc["attention_mask"],# shape [5,128]
        "labels":row["label"]
    }
 
records=[build_example(r) for _,r in subset_df.iterrows()]
hf_dataset=Dataset.from_list(records)
print("Q8 - input_ids shape for item 0:",len(hf_dataset[0]["input_ids"]),"x",len(hf_dataset[0]["input_ids"][0]))
print("Q8 - num tokenized choices:",len(hf_dataset[0]["input_ids"]))

Q8 - input_ids shape for item 0: 5 x 128
Q8 - num tokenized choices: 5


In [13]:
train_df=df.iloc[:32].reset_index(drop=True)
 
def build_example_short(row):
    texts=format_row(row)
    enc=tokenizer(texts,padding="max_length",truncation=True,max_length=64)
    return {
        "input_ids":enc["input_ids"],
        "attention_mask":enc["attention_mask"],
        "labels":row["label"]
    }
 
train_records=[build_example_short(r) for _,r in train_df.iterrows()]
train_dataset=Dataset.from_list(train_records)
 
def collate_fn(batch):
    return {
        "input_ids":torch.tensor([b["input_ids"] for b in batch]),
        "attention_mask":torch.tensor([b["attention_mask"] for b in batch]),
        "labels":torch.tensor([b["labels"] for b in batch])
    }
 
training_args=TrainingArguments(
    output_dir="./tmp_mcq",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    logging_steps=1,
    save_strategy="no",
    report_to=[]
)
 
lora_model_train=get_peft_model(AutoModelForMultipleChoice.from_pretrained(model_name),lora_config)
 
trainer=Trainer(
    model=lora_model_train,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collate_fn
)
 
train_result=trainer.train()
q9_answer=trainer.state.global_step
print("Q9 - final global_step:",q9_answer)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch

Step,Training Loss
1,1.724169
2,1.744465
3,1.665700
4,1.483081


Q9 - final global_step: 4


In [14]:
lora_model_train.eval()
with torch.no_grad():
    infer_outputs=lora_model_train(input_ids=input_ids,attention_mask=attention_mask)
infer_logits=infer_outputs.logits
probs=F.softmax(infer_logits,dim=1)
prob_option_e=probs[0][4].item()
q10_answer=round(prob_option_e,4)
print("Q10 - probability of Option E:",q10_answer)

Q10 - probability of Option E: 0.2004
